# Proyecto - Predicción de Importaciones para Cencosud

Este notebook implementa un flujo de trabajo reproducible para estimar las importaciones de electrodomésticos en Chile con foco en apoyar la planificación logística y de abastecimiento de **Cencosud**.

## Objetivo
Construir un modelo predictivo que permita estimar el comportamiento futuro de las importaciones de electrodomésticos en Chile.

## Variables analizadas
- `valor_cif`
- `peso`
- `cantidad`

## Códigos HS considerados
- `8418`: Refrigeradores
- `8450`: Lavadoras
- `8516`: Microondas / hornos eléctricos
- `8528`: Televisores

## Horizonte de predicción
- `6 meses`

## Modelos a comparar
- ARIMA
- Regresión Lineal
- Random Forest
- XGBoost
- LightGBM

> **Nota importante**: Este notebook fue corregido para trabajar de forma coherente con columnas en **minúscula** y con los archivos:
> - `importaciones_hs_filtrado_estandarizado.csv`
> - `importaciones_hs_filtrado_raw.csv`

## BLOQUE 18 — Forecast a 6 meses del mejor modelo

In [ ]:
forecast_final = []

if "mejores_modelos" in globals():
    for _, row in mejores_modelos.iterrows():
        variable = row["variable"]
        modelo = row["modelo"]

        key_map = {
            "ARIMA": f"{variable}_ARIMA",
            "Regresión Lineal": f"{variable}_RegresionLineal",
            "Random Forest": f"{variable}_RandomForest",
            "XGBoost": f"{variable}_XGBoost",
            "LightGBM": f"{variable}_LightGBM"
        }

        clave = key_map.get(modelo)

        if clave in resultados_forecast:
            temp = resultados_forecast[clave].copy()
            forecast_final.append(temp)

    if len(forecast_final) > 0:
        forecast_final_df = pd.concat(forecast_final, ignore_index=True)
        print("Forecast final a 6 meses:")
        display(forecast_final_df)

        guardar_tabla(forecast_final_df, "forecast_6_meses_mejores_modelos.csv")
    else:
        print("[WARNING] No se generó forecast final.")
else:
    print("[WARNING] No existe mejores_modelos.")

## BLOQUE 19 — Gráficos de validación

In [ ]:
if len(resultados_predicciones) > 0:
    for nombre, pred_df in resultados_predicciones.items():
        plt.figure(figsize=(12, 5))
        plt.plot(pred_df["fecha_mes"], pred_df["y_real"], label="Real")
        plt.plot(pred_df["fecha_mes"], pred_df["y_pred"], label="Predicción")
        plt.title(f"Validación modelo - {nombre}")
        plt.xlabel("Fecha")
        plt.ylabel("Valor")
        plt.legend()
        plt.xticks(rotation=45)
        plt.tight_layout()
        guardar_figura(f"validacion_{nombre}.png")
        plt.show()
else:
    print("[WARNING] No existen predicciones para graficar.")

## BLOQUE 20 — Gráficos del forecast final

In [ ]:
if "forecast_final_df" in globals() and "monthly_total" in globals():
    for variable in forecast_final_df["variable"].unique():
        plt.figure(figsize=(12, 5))

        historico = monthly_total[["fecha_mes", variable]].copy()
        futuro = forecast_final_df[forecast_final_df["variable"] == variable].copy()

        plt.plot(historico["fecha_mes"], historico[variable], label="Histórico")
        plt.plot(futuro["fecha_mes"], futuro["forecast"], label="Forecast")

        modelo_nombre = futuro["modelo"].iloc[0] if len(futuro) > 0 else "N/D"

        plt.title(f"Forecast 6 meses - {variable} | Modelo: {modelo_nombre}")
        plt.xlabel("Fecha")
        plt.ylabel(variable)
        plt.legend()
        plt.xticks(rotation=45)
        plt.tight_layout()
        guardar_figura(f"forecast_{variable}.png")
        plt.show()
else:
    print("[WARNING] No existe forecast_final_df.")

## BLOQUE 21 — Resumen ejecutivo automático

In [ ]:
resumen_ejecutivo = []

resumen_ejecutivo.append("RESUMEN EJECUTIVO")
resumen_ejecutivo.append("=" * 80)
resumen_ejecutivo.append("Objetivo: estimar el comportamiento futuro de las importaciones de electrodomésticos en Chile para apoyar la planificación logística y de abastecimiento de Cencosud.")
resumen_ejecutivo.append("Variables analizadas: valor_cif, peso y cantidad.")
resumen_ejecutivo.append(f"Códigos HS considerados: {', '.join(HS_CODES_OBJETIVO)}.")
resumen_ejecutivo.append(f"Horizonte de predicción: {HORIZONTE_MESES} meses.")
resumen_ejecutivo.append("")

if "monthly_total" in globals():
    resumen_ejecutivo.append(f"Se construyó una serie mensual consolidada con {monthly_total.shape[0]} meses observados.")
else:
    resumen_ejecutivo.append("No fue posible construir la serie mensual consolidada.")

if "monthly_by_hs" in globals():
    resumen_ejecutivo.append(f"Se construyó una serie mensual por HS con {monthly_by_hs.shape[0]} registros.")
else:
    resumen_ejecutivo.append("No fue posible construir la serie mensual por HS.")

if "mejores_modelos" in globals():
    resumen_ejecutivo.append("")
    resumen_ejecutivo.append("Mejor modelo por variable:")
    for _, row in mejores_modelos.iterrows():
        resumen_ejecutivo.append(
            f"- {row['variable']}: {row['modelo']} | RMSE={row['rmse']:.2f} | MAE={row['mae']:.2f} | MAPE={row['mape']:.2f}%"
        )

if "forecast_final_df" in globals():
    resumen_ejecutivo.append("")
    resumen_ejecutivo.append("Forecast a 6 meses:")
    for variable in forecast_final_df["variable"].unique():
        temp = forecast_final_df[forecast_final_df["variable"] == variable].copy()
        modelo = temp["modelo"].iloc[0]
        resumen_ejecutivo.append(f"- Variable: {variable} | Modelo seleccionado: {modelo}")
        for _, r in temp.iterrows():
            resumen_ejecutivo.append(f"  {r['fecha_mes'].strftime('%Y-%m')}: {r['forecast']:.2f}")

texto_resumen = "\n".join(resumen_ejecutivo)
print(texto_resumen)

with open(RESULTS_DIR / "resumen_ejecutivo.txt", "w", encoding="utf-8") as f:
    f.write(texto_resumen)

print(f"\n[OK] Resumen ejecutivo guardado en: {RESULTS_DIR / 'resumen_ejecutivo.txt'}")

## BLOQUE 22 — Validación final de entregables

In [ ]:
print("Validación final de entregables")

entregables = [
    RESULTS_DIR / "resumen_ejecutivo.txt",
    TABLES_DIR / "dataset_final_modelado.csv",
    TABLES_DIR / "dataset_mensual_total.csv",
    TABLES_DIR / "dataset_mensual_por_hs.csv",
    TABLES_DIR / "metricas_comparativas_modelos.csv",
    TABLES_DIR / "mejores_modelos_por_variable.csv",
    TABLES_DIR / "forecast_6_meses_mejores_modelos.csv"
]

for ruta in entregables:
    print(f"{ruta.name}: {'OK' if ruta.exists() else 'NO GENERADO'}")